# H&M Fashion Recommender — Data Acquisition & PreparationThis notebook downloads the [H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations) dataset and prepares it for the recommendation pipeline.**Steps**1. Mount Drive and authenticate with Kaggle2. Download the competition dataset3. Convert the raw CSVs to Parquet (faster I/O, smaller footprint)4. Extract product images for the filtered article setThe dataset is large (~25 GB), so all heavy artifacts are stored on Google Drive and read back as needed.

## 1. SetupImports, Drive mount, and project paths. Paths are centralized in `src/paths.py` so they can be reused across notebooks.

In [ ]:
import osimport gcimport zipfileimport pandas as pdimport pyarrow as paimport pyarrow.parquet as pqfrom tqdm import tqdmimport matplotlib.pyplot as pltimport matplotlib.image as mpimgfrom google.colab import drive, userdatadrive.mount('/content/drive')

In [ ]:
import syssys.path.append("/content/drive/MyDrive/hm_fashion_project/src")from paths import BASE_DIR, RAW_DATA_PATHRAW_DIR = os.path.join(BASE_DIR, "data/raw")BASE_PROCESS = os.path.join(BASE_DIR, "data/processed")os.makedirs(RAW_DIR, exist_ok=True)os.makedirs(BASE_PROCESS, exist_ok=True)print("Project directories ready")

## 2. Download the Dataset from KaggleKaggle credentials are read from Colab secrets (`KAGGLE_KEY`, `KAGGLE_USER`) to avoid hard-coding them.

In [ ]:
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')

In [ ]:
%cd $RAW_DIR!kaggle competitions download -c h-and-m-personalized-fashion-recommendations

## 3. Convert CSVs to ParquetThe dataset ships as CSV. Converting to Parquet gives faster reads and far smaller files, which matters for the 15M+ row transactions table. Each CSV is converted and the dataframe released from memory immediately.

In [ ]:
# extract just the CSV files from the archive%cd $RAW_DATA_PATH!unzip -qn h-and-m-personalized-fashion-recommendations.zip "*.csv" 

In [ ]:
for file in os.listdir(RAW_DATA_PATH):    if file.endswith(".csv"):        df = pd.read_csv(os.path.join(RAW_DATA_PATH, file))        target_name = file.replace(".csv", ".parquet")        df.to_parquet(os.path.join(BASE_PROCESS, target_name), engine='pyarrow', index=False)        del df        gc.collect()        print(f"Converted {file} -> {target_name}")print("All CSV files converted to Parquet")

## 4. Extract Product ImagesOnly articles that survive filtering (the set saved as `articles_final.parquet`) need their images. Extracting just those — rather than all ~105K images — saves significant space and time.Images are stored in the archive as `images/<first 3 digits>/<article_id>.jpg`, and are written out under `data/processed/images_extracted/` preserving that sharded structure.

In [ ]:
# load the filtered article set produced by the filtering notebookdf_articles = pd.read_parquet(os.path.join(BASE_PROCESS, "articles/articles_final.parquet"))needed_ids = set(df_articles['article_id'].astype(str).str.zfill(10).tolist())print(f"Articles requiring images: {len(needed_ids)}")

In [ ]:
zip_path = os.path.join(RAW_DIR, "h-and-m-personalized-fashion-recommendations.zip")extract_to = os.path.join(BASE_PROCESS, "images_extracted")os.makedirs(extract_to, exist_ok=True)def zip_member(article_id):    return f"images/{article_id[:3]}/{article_id}.jpg"needed_members = {zip_member(a) for a in needed_ids}extracted, skipped = 0, 0with zipfile.ZipFile(zip_path, 'r') as z:    available = set(z.namelist())    to_extract = needed_members & available    print(f"Matched {len(to_extract)} images in archive")    for member in tqdm(to_extract):        out_path = os.path.join(extract_to, member.replace("images/", "", 1))        os.makedirs(os.path.dirname(out_path), exist_ok=True)        if os.path.exists(out_path):            skipped += 1            continue        with z.open(member) as src, open(out_path, 'wb') as dst:            dst.write(src.read())        extracted += 1print(f"Extracted: {extracted}, Skipped (already present): {skipped}")print(f"Requested but missing from archive: {len(needed_members - available)}")

## 5. Sanity CheckDisplay one extracted image to confirm the paths and data line up.

In [ ]:
sample_id = next(iter(needed_ids))sample_path = os.path.join(extract_to, sample_id[:3], f"{sample_id}.jpg")if os.path.exists(sample_path):    plt.figure(figsize=(4, 4))    plt.imshow(mpimg.imread(sample_path))    plt.axis('off')    plt.title(f"Article {sample_id}")    plt.show()else:    print(f"Image not found: {sample_path}")